In [ ]:
import os 
# Disable progress bars to avoid Jupyter context errors
os.environ['HF_DATASETS_DISABLE_PROGRESS_BAR'] = '1'
import re          
import random      
import logging     
import warnings    
from typing import List, Dict, Optional, Tuple  
from dataclasses import dataclass, field  

import numpy as np  
import torch 
from trl import (
    GRPOConfig,   
    GRPOTrainer 
)
from transformers import (
    AutoTokenizer,  
    AutoModelForCausalLM,    
)

from datasets import load_dataset, load_from_disk


from utils import (
    setup_logging,  
    load_and_explore_gsm8k_dataset, 
    prepare_dataset, 
    GSM8KEvaluationCallback, 
    evaluate_and_compare              
)


warnings.filterwarnings('ignore')
random.seed(42)    
np.random.seed(42)
torch.manual_seed(42)
torch.use_deterministic_algorithms(True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger, log_file = setup_logging(device=device)

print("✅ libraries imported")

In [ ]:
# ============================================
# Define Training Configuration Class
# ============================================

@dataclass
class TrainingConfig:
    """
    Configuration for GRPO training.
    
    This class holds all the settings (hyperparameters) for training.
    Think of it as a control panel with all the knobs and switches.
    
    Each parameter has:
    - A default value (recommended value)
    - A description (what it does)
    - A type (what kind of value it expects)
    """
    
    # ========== MODEL SETTINGS ==========
    # Which model to use and where to save it
    
    model_name: str = field(
        default="/app/models/deepseek-math-7b-base",
        metadata={"help": "The pre-trained model to start with."}
    )
    
    output_dir: str = field(
        default="./grpo_finetuned_model",
        metadata={"help": "Where to save the trained model. Like a 'Save As' location."}
    )
    
    # ========== TRAINING DURATION ==========
    # How long to train for
    
    num_train_epochs: int = field(
        default=5,
        metadata={"help": "How many times to go through the training data. More = more learning."}
    )
    
    # ========== BATCH SETTINGS ==========
    # How many examples to process at once
    
    per_device_train_batch_size: int = field(
        default=2,
        metadata={"help": "How many problems to process at once. Limited by GPU memory."}
    )
    
    gradient_accumulation_steps: int = field(
        default=32,
        metadata={"help": "Accumulate gradients over multiple batches. Simulates larger batch size."}
    )
    # Effective batch size = per_device_train_batch_size * gradient_accumulation_steps = 64
    
    # ========== LEARNING SETTINGS ==========
    # How fast the model learns
    
    learning_rate: float = field(
        default=5e-6,  # 0.000005 in decimal
        metadata={"help": "How big of a step to take when learning. Too high = unstable."}
    )
    
    # ========== GRPO SPECIFIC SETTINGS ==========
    # Settings unique to GRPO algorithm
    
    num_generations: int = field(
        default=12,
        metadata={"help": "How many different answers to generate per question. More = better comparison."}
    )
    
    temperature: float = field(
        default=0.8,
        metadata={"help": "Controls randomness. 0.0 = always same answer, 1.0 = very random."}
    )
    
    max_new_tokens: int = field(
        default=400,
        metadata={"help": "Maximum length of generated answers. Needs to be long enough for full solutions."}
    )
    
    # ========== DATA SETTINGS ==========
    # How to handle the dataset
    
    max_prompt_length: int = field(
        default=512,
        metadata={"help": "Maximum length of input questions in tokens."}
    )
    
    train_split_ratio: float = field(
        default=0.8,
        metadata={"help": "What fraction of data to use for training (rest for validation)."}
    )
    
    # ========== MONITORING SETTINGS ==========
    # How often to check progress
    
    eval_steps: int = field(
        default=20,
        metadata={"help": "Evaluate model every N steps to check progress."}
    )
    
    save_steps: int = field(
        default=20,
        metadata={"help": "Save a checkpoint every N steps (for recovery if training stops)."}
    )
    
    logging_steps: int = field(
        default=20,
        metadata={"help": "Log training metrics every N steps."}
    )
    
    save_total_limit: int = field(
        default=3,
        metadata={"help": "Keep only the N most recent checkpoints to save disk space."}
    )
    
    # ========== OTHER SETTINGS ==========
    
    seed: int = field(
        default=42,
        metadata={"help": "Random seed for reproducibility."}
    )
    
    use_8bit: bool = field(
        default=False,
        metadata={"help": "Load model in 8-bit mode to save memory (slightly less accurate)."}
    )

print("✅ Configuration class defined!")

In [ ]:
# ============================================
# Create and Display Configuration
# ============================================

# Create an instance of your configuration
# This uses all the default values defined above
config = TrainingConfig()

# Display all configuration values
print("TRAINING CONFIGURATION")
print("="*50)

# Group settings by category for easier reading
print("\nModel Settings:")
print(f"  Model: {config.model_name}")
print(f"  Output directory: {config.output_dir}")

print("\nTraining Duration:")
print(f"  Epochs: {config.num_train_epochs}")
print(f"  Batch size per device: {config.per_device_train_batch_size}")
print(f"  Gradient accumulation: {config.gradient_accumulation_steps}")
print(f"  Effective batch size: {config.per_device_train_batch_size * config.gradient_accumulation_steps}")

print("\nGRPO Settings:")
print(f"  Generations per prompt: {config.num_generations}")
print(f"  Temperature: {config.temperature}")
print(f"  Max new tokens: {config.max_new_tokens}")
print(f"  Learning rate: {config.learning_rate}")

print("\nMonitoring:")
print(f"  Evaluate every: {config.eval_steps} steps")
print(f"  Save every: {config.save_steps} steps")
print(f"  Log every: {config.logging_steps} steps")

# Calculate approximate training time
print("\nEstimated Training Info:")
print(f"  This configuration will generate {config.num_generations} answers per question")
print(f"  The model will learn by comparing these answers")